In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import pandas as pd

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
print(device)

cuda


In [5]:
file_path_train = '/content/drive/My Drive/001.PhD/Smartphone/traintest/trainDataset2.csv'

In [6]:
df = pd.read_csv(file_path_train)

In [7]:
df.head()

,sequence
0,"android.net.nsd.STATE_CHANGED, android.net.con..."
1,"android.net.nsd.STATE_CHANGED, android.net.con..."
2,"android.intent.action.DREAMING_STARTED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.intent.action.DREAMING_STARTED, androi..."


In [8]:
sequence_list = df['sequence'].tolist()

In [ ]:
# print(sequence_list[0])

In [9]:
sequence_list = sequence_list[:20000]
print(len(sequence_list))

20000


In [10]:
trainDataset = []
for sequence in sequence_list:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    trainDataset.append(result)

In [ ]:
# trainDataset[0]

In [ ]:
# trainDataset[0][-1]

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [12]:
# Build vocabulary
all_events = set(e for seq in trainDataset for e in seq)
event2idx = {event: idx for idx, event in enumerate(all_events)}
idx2event = {idx: event for event, idx in event2idx.items()}

vocab_size = len(event2idx)

In [13]:
class ContextDataset(Dataset):
    def __init__(self, dataset, event2idx):
        self.dataset = dataset
        self.event2idx = event2idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        seq = self.dataset[idx]
        input_seq = torch.tensor(
            [self.event2idx[e] for e in seq[:-1]],
            dtype=torch.long
        )
        target = torch.tensor(
            self.event2idx[seq[-1]],
            dtype=torch.long
        )
        return input_seq, target

In [14]:
def collate_fn(batch):
    sequences, targets = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])

    padded_seqs = nn.utils.rnn.pad_sequence(
        sequences, batch_first=True
    )

    return padded_seqs, lengths, torch.tensor(targets)

In [15]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(
            embed_dim, hidden_dim, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, lengths):
        # x: [batch, seq_len]
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, hidden = self.rnn(packed)
        # hidden: [1, batch, hidden_dim]

        hidden = hidden.squeeze(0)
        logits = self.fc(hidden)
        return logits

In [16]:
embed_dim = 64
hidden_dim = 128
batch_size = 16
epochs = 10
lr = 0.001

In [17]:
dataset = ContextDataset(trainDataset, event2idx)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

model = VanillaRNN(vocab_size, embed_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [18]:
for epoch in range(epochs):
    total_loss = 0.0

    for inputs, lengths, targets in dataloader:
        optimizer.zero_grad()

        outputs = model(inputs, lengths)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")

Epoch 1, Loss: 1.1331
Epoch 2, Loss: 0.9573
Epoch 3, Loss: 0.9252
Epoch 4, Loss: 0.9086
Epoch 5, Loss: 0.8947
Epoch 6, Loss: 0.8884
Epoch 7, Loss: 0.8782
Epoch 8, Loss: 0.8740
Epoch 9, Loss: 0.8703
Epoch 10, Loss: 0.8651


In [19]:
file_path_test = '/content/drive/My Drive/001.PhD/Smartphone/traintest/testDataset2.csv'

In [20]:
dfTest = pd.read_csv(file_path_test)

In [21]:
dfTest.head()

,sequence
0,"android.net.wifi.RSSI_CHANGED, android.intent...."
1,"android.intent.action.DREAMING_STARTED, androi..."
2,"android.intent.action.DREAMING_STOPPED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.net.wifi.RSSI_CHANGED, android.intent...."


In [22]:
sequence_list_test = dfTest['sequence'].tolist()

In [23]:
sequence_list_test = sequence_list_test[:8000]

In [24]:
testDataset = []
for sequence in sequence_list_test:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    testDataset.append(result)

In [25]:
test_dataset = ContextDataset(testDataset, event2idx)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [26]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, lengths, targets in test_loader:
        outputs = model(inputs, lengths)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(targets.tolist())
        y_pred.extend(preds.tolist())

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [28]:
precision_micro = precision_score(
    y_true, y_pred, average="micro", zero_division=0
)

recall_micro = recall_score(
    y_true, y_pred, average="micro", zero_division=0
)

f1_macro = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

f1_weighted = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

In [29]:
print(f"Precision (micro): {precision_micro:.4f}")
print(f"Recall (micro):    {recall_micro:.4f}")
print(f"F1-score (Macro):  {f1_macro:.4f}")
print(f"F1-score (Weighted): {f1_weighted:.4f}")

Precision (micro): 0.7585
Recall (micro):    0.7585
F1-score (Macro):  0.2997
F1-score (Weighted): 0.7450
